In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [2]:
def getSparkSession():
    spark = SparkSession.builder.appName('weather').getOrCreate()
    return spark

def load_data(filepath, spark):
    df = spark.read.csv(filepath, header=True, inferSchema=True)
    return df

In [3]:
filepath='D:/spark/datasets/weather.csv'

spark = getSparkSession()
df = load_data(filepath, spark)

In [6]:
def get_columns(df):
    collist=[]
    for col in df.columns:
        collist.append(col)
    collist.remove('YEAR')
    return collist
    
collist = get_columns(df)

In [7]:
# 1. Add column average temperature for each year 
def query_1(df, spark):
    total_sum = 0
    for m in collist:
        total_sum+=F.col(m)
    df1 = df.withColumn('AVG_TEMP', round(total_sum/len(collist), 2))
    return df1
df = query_1(df, spark)
df.head(3)

[Row(YEAR=1901, JAN=17.99, FEB=19.43, MAR=23.49, APR=26.41, MAY=28.28, JUN=28.6, JUL=27.49, AUG=26.98, SEP=26.26, OCT=25.08, NOV=21.73, DEC=18.95, AVG_TEMP=24.22),
 Row(YEAR=1902, JAN=19.0, FEB=20.39, MAR=24.1, APR=26.54, MAY=28.68, JUN=28.44, JUL=27.29, AUG=27.05, SEP=25.95, OCT=24.37, NOV=21.33, DEC=18.78, AVG_TEMP=24.33),
 Row(YEAR=1903, JAN=18.32, FEB=19.79, MAR=22.46, APR=26.03, MAY=27.93, JUN=28.41, JUL=28.04, AUG=26.63, SEP=26.34, OCT=24.57, NOV=20.96, DEC=18.29, AVG_TEMP=23.98)]

In [8]:
# 2. Add row average temperature of each month
def query_2(df):
    df1 = df.agg(*[round(F.avg(c), 2).alias(c) for c in collist])
    df_string_year = df.withColumn("YEAR", F.col("YEAR").cast("string"))
    df_avg = df1.withColumn("YEAR", F.lit("Average"))
    return df_avg
df1 = query_2(df)
df1.show()

+-----+-----+-----+-----+-----+----+-----+-----+-----+-----+-----+-----+-------+
|  JAN|  FEB|  MAR|  APR|  MAY| JUN|  JUL|  AUG|  SEP|  OCT|  NOV|  DEC|   YEAR|
+-----+-----+-----+-----+-----+----+-----+-----+-----+-----+-----+-----+-------+
|18.42|20.14|23.43|26.51|28.39|28.3|27.37|26.94|26.34|24.74|21.77|19.17|Average|
+-----+-----+-----+-----+-----+----+-----+-----+-----+-----+-----+-----+-------+



In [9]:
# 3. Add columns Min and Max for each year with corresponding values 
def query_3(df):
    df1 = df.withColumn('MIN_TEMP', F.least(*collist)).withColumn('MAX_TEMP', F.greatest(*collist))
    return df1
df2 = query_3(df)
df2.show()

+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+--------+--------+--------+
|YEAR|  JAN|  FEB|  MAR|  APR|  MAY|  JUN|  JUL|  AUG|  SEP|  OCT|  NOV|  DEC|AVG_TEMP|MIN_TEMP|MAX_TEMP|
+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+--------+--------+--------+
|1901|17.99|19.43|23.49|26.41|28.28| 28.6|27.49|26.98|26.26|25.08|21.73|18.95|   24.22|   17.99|    28.6|
|1902| 19.0|20.39| 24.1|26.54|28.68|28.44|27.29|27.05|25.95|24.37|21.33|18.78|   24.33|   18.78|   28.68|
|1903|18.32|19.79|22.46|26.03|27.93|28.41|28.04|26.63|26.34|24.57|20.96|18.29|   23.98|   18.29|   28.41|
|1904|17.77|19.39|22.95|26.73|27.83|27.85|26.84|26.73|25.84|24.36|21.07|18.84|   23.85|   17.77|   27.85|
|1905| 17.4|17.79|21.78|24.84|28.32|28.69|27.67|27.47|26.29|26.16|22.07|18.71|   23.93|    17.4|   28.69|
|1906| 17.5|19.14|22.21|26.53|29.06|28.02|27.46|26.82|26.23|24.75|21.93|19.55|    24.1|    17.5|   29.06|
|1907|19.27|19.42|22.03|26.98|27.52|27.66|27.2

In [10]:
# 4. Create dataframe called 'decade', storing average temperatures of each month in the respective decade. Decade 1910 is from 1901 to 1910

df3 = df.withColumn('Decade', (floor((col("YEAR") - 1) / 10) + 1) * 10)\
    .groupBy('Decade')\
    .agg(round(avg("JAN")).alias("JAN"),
    avg("FEB").alias("FEB"),
    avg("MAR").alias("MAR"),
    avg("APR").alias("APR"),
    avg("MAY").alias("MAY"),
    avg("JUN").alias("JUN"),
    avg("JUL").alias("JUL"),
    avg("AUG").alias("AUG"),
    avg("SEP").alias("SEP"),
    avg("OCT").alias("OCT"),
    avg("NOV").alias("NOV"),
    avg("DEC").alias("DEC")).orderBy('Decade')

df3.show()



+------+----+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|Decade| JAN|               FEB|               MAR|               APR|               MAY|               JUN|               JUL|               AUG|               SEP|               OCT|               NOV|               DEC|
+------+----+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|  1910|18.0|19.384999999999998|22.825000000000003|            26.284|28.201999999999998| 28.15500000000001|            27.272|26.750999999999998|            26.036|            24.667|            21.499|18.663999999999998|
|  1920|18.0|            19.617|22.796000000000003|26.008999999999997|28.086999999999996|            28.036|

In [13]:
# 5. Which year is the hottest year? (Based on avg temp)

df5 = df2.select('YEAR', 'AVG_TEMP').orderBy('AVG_TEMP', ascending=False).limit(3)
df5.show()

+----+--------+
|YEAR|AVG_TEMP|
+----+--------+
|2016|   26.45|
|2017|   26.29|
|1995|   25.28|
+----+--------+



In [14]:
# 6. Which year is the coldest year? (Based on avg temp)

df6 = df2.select('YEAR', 'AVG_TEMP').orderBy('AVG_TEMP', ascending=True).limit(3)
df6.show()

+----+--------+
|YEAR|AVG_TEMP|
+----+--------+
|1917|   23.57|
|1950|   23.74|
|1975|   23.74|
+----+--------+



In [16]:
# 7. Find out which years recorded minimum and maximum temperatures. (Not based on avg temp)

df7 = df2.withColumn(
    "Min_Temp",
    least("JAN", "FEB", "MAR", "APR", "MAY", "JUN",
          "JUL", "AUG", "SEP", "OCT", "NOV", "DEC")
).withColumn(
    "Max_Temp",
    greatest("JAN", "FEB", "MAR", "APR", "MAY", "JUN",
             "JUL", "AUG", "SEP", "OCT", "NOV", "DEC")
)

df7.select('YEAR', 'Min_Temp').orderBy('Min_Temp', ascending=True).limit(2).show()
df7.select('YEAR', 'Max_Temp').orderBy('Max_Temp', ascending=False).limit(2).show()

+----+--------+
|YEAR|Min_Temp|
+----+--------+
|1918|   17.25|
|1945|   17.33|
+----+--------+

+----+--------+
|YEAR|Max_Temp|
+----+--------+
|1921|   30.78|
|2017|   30.47|
+----+--------+



In [20]:
from pyspark.sql.functions import min, max, col

months = [
    "JAN", "FEB", "MAR", "APR", "MAY", "JUN",
    "JUL", "AUG", "SEP", "OCT", "NOV", "DEC"
]

# Calculate the minimum and maximum temperature for each month
for month in months:
    df.select(
        min(col(month)).alias("Minimum"),
        max(col(month)).alias("Maximum"),
        round((max(col(month)) - min(col(month)))).alias("Temperature_Range")
    ).show()

+-------+-------+-----------------+
|Minimum|Maximum|Temperature_Range|
+-------+-------+-----------------+
|  17.25|  20.92|              4.0|
+-------+-------+-----------------+

+-------+-------+-----------------+
|Minimum|Maximum|Temperature_Range|
+-------+-------+-----------------+
|  17.79|  23.58|              6.0|
+-------+-------+-----------------+

+-------+-------+-----------------+
|Minimum|Maximum|Temperature_Range|
+-------+-------+-----------------+
|  21.78|  26.61|              5.0|
+-------+-------+-----------------+

+-------+-------+-----------------+
|Minimum|Maximum|Temperature_Range|
+-------+-------+-----------------+
|  24.84|  29.56|              5.0|
+-------+-------+-----------------+

+-------+-------+-----------------+
|Minimum|Maximum|Temperature_Range|
+-------+-------+-----------------+
|  26.97|  30.78|              4.0|
+-------+-------+-----------------+

+-------+-------+-----------------+
|Minimum|Maximum|Temperature_Range|
+-------+-------+------

In [ ]:
|